# MRI Preprocessing Pipeline
### Tumor segmentation + feature extraction project

**Data:** 45 real patients (44 used — 1 excluded, see Step 0), single-modality
contrast-enhanced T1 (T1+GD) MRI with radiologist-drawn tumor masks.

**Pipeline stages:**
1. Patient/scan selection
2. Orientation standardization
3. Bias field correction
4. Histogram matching (intensity normalization)
5. Registration to MNI template
6. Skull stripping

Each stage reads the previous stage's saved output and writes its own —
nothing is recomputed from raw files after Step 1, and every step can be
safely re-run (it skips patients already completed).


## Setup — imports

In [ ]:
%matplotlib inline
import os
import glob
import gc
import shutil
import re

import pandas as pd
import numpy as np
import ants
import SimpleITK as sitk
import matplotlib.pyplot as plt
from antspynet.utilities import brain_extraction

print(f'AntsPy version   = {ants.__version__}')
print(f'SimpleITK version = {sitk.__version__}')

BASE_DIR = r"D:\MRI_ALL_NOTEBOOKS"
RAW_DIR = r"D:\mri"
PREP_DIR = os.path.join(BASE_DIR, "preprocessed")
TEMPLATE_PATH = os.path.join(BASE_DIR, "assets", "templates",
                              "mni_icbm152_t1_tal_nlin_sym_09a.nii")
os.makedirs(PREP_DIR, exist_ok=True)
print(f'Project folder    = {BASE_DIR}')

## Step 0 — Patient and scan selection

The raw dataset has 194 scans across 45 patients: each patient has a
different number of timepoints and different sequence names, so a single
representative scan is selected per patient before any processing begins.

**Selection rule:** the earliest contrast-enhanced (`+GD`) scan that has a
matching tumor mask. This avoids using multiple timepoints from the same
patient (which would leak patient identity across train/validation splits
later) and avoids scans with no ground-truth annotation.

**Excluded: patient `040019`.** All of this patient's available scans were
`MultiPlanar Reconstruction (MPR)` images — a derived/reformatted scan type,
not a standard contrast acquisition — and were roughly 6x larger than a
typical patient's file (~128 MB vs ~20 MB), which made them unprocessable on
this machine. Verified: no `+GD`-type scan existed for this patient in the
raw data, so there was no valid alternative to select instead.


In [ ]:
def build_manifest(raw_dir):
    """One row per scan found in raw_dir, paired with its mask if one exists."""
    rows = []
    for img_path in glob.glob(os.path.join(raw_dir, "**", "*_img.nii"), recursive=True):
        mask_path = img_path.replace("_img.nii", "_msk.nii")
        fname = os.path.basename(img_path)
        m = re.match(r"(\d+)_(\d+)_(.+)_img\.nii", fname)
        if not m:
            continue
        patient_id, scan_date, sequence = m.groups()
        rows.append({
            "patient_id": patient_id, "scan_date": scan_date, "sequence": sequence,
            "img_path": img_path,
            "mask_path": mask_path if os.path.exists(mask_path) else None,
        })
    return pd.DataFrame(rows)


def select_scan_per_patient(manifest):
    """One scan per patient: earliest +GD scan that has a matching mask.
    Falls back to any masked scan if no +GD scan has one."""
    chosen, skipped = [], []
    for patient_id, group in manifest.groupby("patient_id"):
        has_mask = group[group["mask_path"].notna()]
        if len(has_mask) == 0:
            skipped.append(patient_id)
            continue
        gd_scans = has_mask[has_mask["sequence"].str.contains("GD", case=False)]
        candidates = gd_scans if len(gd_scans) > 0 else has_mask
        chosen.append(candidates.sort_values("scan_date").iloc[0])
    if skipped:
        print(f"Excluded (no mask available): {skipped}")
    return pd.DataFrame(chosen).reset_index(drop=True)


manifest = build_manifest(RAW_DIR)
selected = select_scan_per_patient(manifest)

# Excluded: oversized (~128MB) non-contrast MPR scans, no +GD alternative available
selected = selected[selected["patient_id"] != "040019"].reset_index(drop=True)

print(f"Found {len(manifest)} total scans across {manifest['patient_id'].nunique()} patients")
print(f"Selected {len(selected)} patients for preprocessing")
selected.head()

## Step 1 — Orientation standardization

MRI scanners don't guarantee consistent orientation labeling across sites
or protocols. Every image and mask is reoriented to a single convention
(`IAL`) using ANTsPy, so "left", "up", etc. mean the same physical direction
for every patient, and image/mask pairs stay spatially aligned with each
other.


In [ ]:
def explore_3D_array(arr):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    mid = [arr.shape[i] // 2 for i in range(3)]
    axes[0].imshow(arr[mid[0], :, :], cmap='gray'); axes[0].set_title('Sagittal')
    axes[1].imshow(arr[:, mid[1], :], cmap='gray'); axes[1].set_title('Coronal')
    axes[2].imshow(arr[:, :, mid[2]], cmap='gray'); axes[2].set_title('Axial')
    plt.show()

STEP1_DIR = os.path.join(PREP_DIR, "step1_reoriented")
os.makedirs(STEP1_DIR, exist_ok=True)

already_done = set(os.path.basename(f).replace("_img_step1.nii", "")
                    for f in glob.glob(os.path.join(STEP1_DIR, "*_img_step1.nii")))
remaining = selected[~selected["patient_id"].isin(already_done)].reset_index(drop=True)
print(f"{len(remaining)} patients remaining for Step 1")

for i, row in remaining.iterrows():
    patient_id = row["patient_id"]
    img = ants.image_read(row["img_path"], reorient='IAL')
    mask = ants.image_read(row["mask_path"], reorient='IAL')
    ants.image_write(img, os.path.join(STEP1_DIR, f"{patient_id}_img_step1.nii"))
    ants.image_write(mask, os.path.join(STEP1_DIR, f"{patient_id}_mask_step1.nii"))
    if i == 0:
        print(f"Patient {patient_id} - orientation: {img.get_orientation()}, shape: {img.numpy().shape}")
        explore_3D_array(img.numpy())

print(f"Step 1 complete - {len(glob.glob(os.path.join(STEP1_DIR, '*_img_step1.nii')))}/{len(selected)} patients done.")

## Step 2 — Bias field correction

MRI scans have smooth, low-frequency intensity variation across the image
caused by imperfections in the scanner's magnetic field (not by anatomy).
N4 bias field correction estimates and removes this, so intensity
differences in the image reflect real tissue differences rather than
scanner artifacts. Applied to the image only — the mask is carried forward
unchanged, since it's a binary label, not intensity data.


In [ ]:
STEP2_DIR = os.path.join(PREP_DIR, "step2_bias_corrected")
os.makedirs(STEP2_DIR, exist_ok=True)

already_done = set(os.path.basename(f).replace("_img_step2.nii", "")
                    for f in glob.glob(os.path.join(STEP2_DIR, "*_img_step2.nii")))
remaining = selected[~selected["patient_id"].isin(already_done)].reset_index(drop=True)
print(f"{len(remaining)} patients remaining for Step 2")

failed_patients = []
for i, row in remaining.iterrows():
    patient_id = row["patient_id"]
    print(f"[{i+1}/{len(remaining)}] {patient_id}...")
    try:
        img_in = os.path.join(STEP1_DIR, f"{patient_id}_img_step1.nii")
        mask_in = os.path.join(STEP1_DIR, f"{patient_id}_mask_step1.nii")

        raw_img_sitk = sitk.ReadImage(img_in, sitk.sitkFloat32)
        transformed = sitk.RescaleIntensity(raw_img_sitk, 0, 255)
        head_mask = sitk.LiThreshold(transformed, 0, 1)
        del transformed

        shrinkFactor = 4
        inputImage = sitk.Shrink(raw_img_sitk, [shrinkFactor] * raw_img_sitk.GetDimension())
        maskImage = sitk.Shrink(head_mask, [shrinkFactor] * raw_img_sitk.GetDimension())
        del head_mask

        bias_corrector = sitk.N4BiasFieldCorrectionImageFilter()
        bias_corrector.Execute(inputImage, maskImage)
        del inputImage, maskImage

        log_bias_field = bias_corrector.GetLogBiasFieldAsImage(raw_img_sitk)
        bias_field = sitk.Exp(log_bias_field)
        del log_bias_field, bias_corrector

        corrected = raw_img_sitk / bias_field
        del bias_field, raw_img_sitk

        sitk.WriteImage(corrected, os.path.join(STEP2_DIR, f"{patient_id}_img_step2.nii"))
        shutil.copyfile(mask_in, os.path.join(STEP2_DIR, f"{patient_id}_mask_step2.nii"))
        del corrected

    except Exception as e:
        print(f"   FAILED on {patient_id}: {e}")
        failed_patients.append(patient_id)
    gc.collect()

print(f"Step 2 complete. Failed this run: {failed_patients if failed_patients else 'none'}")
print("(If any failed, just re-run this cell - it will only retry what's missing.)")

## Step 3 — Histogram matching

Even after bias correction, overall intensity scale still varies between
patients and scanners (e.g. "bright" means a different raw number on
different machines). Histogram matching reshapes each patient's intensity
distribution to match a reference (the MNI template), so intensity values
are comparable across the whole cohort — required both for stable model
training and for meaningful feature extraction later.


In [ ]:
STEP3_DIR = os.path.join(PREP_DIR, "step3_histogram_matched")
os.makedirs(STEP3_DIR, exist_ok=True)

already_done = set(os.path.basename(f).replace("_img_step3.nii", "")
                    for f in glob.glob(os.path.join(STEP3_DIR, "*_img_step3.nii")))
remaining = selected[~selected["patient_id"].isin(already_done)].reset_index(drop=True)
print(f"{len(remaining)} patients remaining for Step 3")

template_img_sitk = sitk.ReadImage(TEMPLATE_PATH, sitk.sitkFloat32)
template_img_sitk = sitk.DICOMOrient(template_img_sitk, 'IAL')

failed_patients = []
for i, row in remaining.iterrows():
    patient_id = row["patient_id"]
    print(f"[{i+1}/{len(remaining)}] {patient_id}...")
    try:
        img_in = os.path.join(STEP2_DIR, f"{patient_id}_img_step2.nii")
        mask_in = os.path.join(STEP2_DIR, f"{patient_id}_mask_step2.nii")
        img_sitk = sitk.ReadImage(img_in, sitk.sitkFloat32)
        transformed = sitk.HistogramMatching(img_sitk, template_img_sitk)
        sitk.WriteImage(transformed, os.path.join(STEP3_DIR, f"{patient_id}_img_step3.nii"))
        shutil.copyfile(mask_in, os.path.join(STEP3_DIR, f"{patient_id}_mask_step3.nii"))
        del img_sitk, transformed
    except Exception as e:
        print(f"   FAILED on {patient_id}: {e}")
        failed_patients.append(patient_id)
    gc.collect()

print(f"Step 3 complete. Failed this run: {failed_patients if failed_patients else 'none'}")

## Step 4 — Registration to MNI template

Every patient's brain is warped into the same standardized coordinate space
(the MNI152 template) using deformable (SyN) registration. This makes tumor
*location* and *shape* comparable across patients — essential for this
project's goal of relating tumor characteristics to their primary source.

Images are resampled to the template's resolution (position unchanged)
before registration, since raw scan resolution varies considerably across
patients and very high-resolution scans were computationally infeasible to
register directly. The tumor mask is warped using the exact same transform
as the image, with **nearest-neighbor interpolation** — this is essential:
the default (linear) interpolation would blur a binary mask's edges into
fractional values instead of preserving clean tumor/non-tumor labels.


In [ ]:
STEP4_DIR = os.path.join(PREP_DIR, "step4_registered")
TRANSFORM_DIR = os.path.join(PREP_DIR, "step4_transforms")
os.makedirs(STEP4_DIR, exist_ok=True)
os.makedirs(TRANSFORM_DIR, exist_ok=True)

already_done = set(os.path.basename(f).replace("_img_step4.nii", "")
                    for f in glob.glob(os.path.join(STEP4_DIR, "*_img_step4.nii")))
remaining = selected[~selected["patient_id"].isin(already_done)].reset_index(drop=True)
print(f"{len(remaining)} patients remaining for Step 4")

template_img_ants = ants.image_read(TEMPLATE_PATH, reorient='IAL')

failed_patients = []
for i, row in remaining.iterrows():
    patient_id = row["patient_id"]
    print(f"[{i+1}/{len(remaining)}] Registering {patient_id}...")
    try:
        img_in = os.path.join(STEP3_DIR, f"{patient_id}_img_step3.nii")
        mask_in = os.path.join(STEP3_DIR, f"{patient_id}_mask_step3.nii")

        img_ants = ants.image_read(img_in)
        mask_ants = ants.image_read(mask_in)

        # Resample to template RESOLUTION only - keeps native position,
        # registration still starts from a normal alignment
        img_ants = ants.resample_image(img_ants, template_img_ants.spacing, False, 0)   # linear
        mask_ants = ants.resample_image(mask_ants, template_img_ants.spacing, False, 1)  # nearest neighbor

        transformation = ants.registration(
            fixed=template_img_ants, moving=img_ants,
            type_of_transform='SyN', verbose=False,
        )
        registered_img = transformation['warpedmovout']

        registered_mask = ants.apply_transforms(
            moving=mask_ants, fixed=template_img_ants,
            transformlist=transformation['fwdtransforms'],
            interpolator='nearestNeighbor',
        )

        registered_img.to_file(os.path.join(STEP4_DIR, f"{patient_id}_img_step4.nii"))
        registered_mask.to_file(os.path.join(STEP4_DIR, f"{patient_id}_mask_step4.nii"))

        patient_tf_dir = os.path.join(TRANSFORM_DIR, patient_id)
        os.makedirs(patient_tf_dir, exist_ok=True)
        for j, tf in enumerate(transformation['fwdtransforms']):
            shutil.copyfile(tf, os.path.join(patient_tf_dir, f"fwdtransform_{j}{os.path.splitext(tf)[1]}"))

        del img_ants, mask_ants, transformation, registered_img, registered_mask
    except Exception as e:
        print(f"   FAILED on {patient_id}: {e}")
        failed_patients.append(patient_id)
    gc.collect()

print(f"Step 4 complete. Failed this run: {failed_patients if failed_patients else 'none'}")

## Step 5 — Skull stripping (brain extraction)

Removes non-brain tissue (skull, scalp) using a deep-learning brain
extraction model (ANTsPyNet). This keeps downstream segmentation and
feature extraction focused on brain tissue only. The same brain mask is
applied to the tumor mask as well, so both stay spatially consistent.

This is the **final preprocessing step** — its output is what feeds into
segmentation.


In [ ]:
def explore_3D_array_with_mask_contour(arr, mask):
    mid = arr.shape[0] // 2
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(arr[mid], cmap='gray')
    ax.contour(mask[mid], colors='lime', linewidths=1)
    plt.show()

STEP5_DIR = os.path.join(PREP_DIR, "step5_skull_stripped")
os.makedirs(STEP5_DIR, exist_ok=True)

already_done = set(os.path.basename(f).replace("_img_step5.nii", "")
                    for f in glob.glob(os.path.join(STEP5_DIR, "*_img_step5.nii")))
remaining = selected[~selected["patient_id"].isin(already_done)].reset_index(drop=True)
print(f"{len(remaining)} patients remaining for Step 5")

failed_patients = []
for i, row in remaining.iterrows():
    patient_id = row["patient_id"]
    print(f"[{i+1}/{len(remaining)}] Skull-stripping {patient_id}...")
    try:
        img_in = os.path.join(STEP4_DIR, f"{patient_id}_img_step4.nii")
        mask_in = os.path.join(STEP4_DIR, f"{patient_id}_mask_step4.nii")

        img_ants = ants.image_read(img_in)
        tumor_mask_ants = ants.image_read(mask_in)

        prob_brain_mask = brain_extraction(img_ants, modality="t1", verbose=False)
        brain_mask = ants.get_mask(prob_brain_mask, low_thresh=0.5)

        masked_img = ants.mask_image(img_ants, brain_mask)
        masked_tumor_mask = ants.mask_image(tumor_mask_ants, brain_mask)

        masked_img.to_file(os.path.join(STEP5_DIR, f"{patient_id}_img_step5.nii"))
        masked_tumor_mask.to_file(os.path.join(STEP5_DIR, f"{patient_id}_mask_step5.nii"))

        if i == 0:
            explore_3D_array_with_mask_contour(img_ants.numpy(), brain_mask.numpy())

        del img_ants, tumor_mask_ants, prob_brain_mask, brain_mask, masked_img, masked_tumor_mask
    except Exception as e:
        print(f"   FAILED on {patient_id}: {e}")
        failed_patients.append(patient_id)
    gc.collect()

print(f"Step 5 complete. Failed this run: {failed_patients if failed_patients else 'none'}")
print("Preprocessing pipeline finished - output ready in step5_skull_stripped for segmentation.")

## Verification — final sanity checks

Confirms the full pipeline end to end: correct patient count, and visual
confirmation that the tumor mask still lands on the tumor after every
transformation (reorientation, bias correction, histogram matching,
registration, skull stripping).


In [ ]:
# File count check
files = glob.glob(os.path.join(STEP5_DIR, "*_img_step5.nii"))
print(f"Final preprocessed patients: {len(files)} / {len(selected)}")

# Visual check across a few patients with different tumor sizes
for patient_id in ["010005", "040006", "040094"]:
    img_path = os.path.join(STEP5_DIR, f"{patient_id}_img_step5.nii")
    mask_path = os.path.join(STEP5_DIR, f"{patient_id}_mask_step5.nii")
    if not os.path.exists(img_path):
        continue
    img = sitk.GetArrayFromImage(sitk.ReadImage(img_path))
    mask = sitk.GetArrayFromImage(sitk.ReadImage(mask_path))
    best_slice = (mask > 0).sum(axis=(1, 2)).argmax()

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img[best_slice], cmap='gray'); axes[0].set_title(f"{patient_id} - image")
    axes[1].imshow(img[best_slice], cmap='gray')
    axes[1].imshow(mask[best_slice], cmap='Reds', alpha=0.4); axes[1].set_title("with tumor mask")
    plt.show()